# Amazon ML Challenge: Business Entity Resolution Pipeline
This notebook breaks down the machine learning pipeline step-by-step so you can run it interactively.

## 1. Imports & Configuration
First, we import the required libraries and the custom modules from our `src/` directory.

In [ ]:
import sys, os, gc, random
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm  # Use notebook version of tqdm

# Import custom modules from src
from src import config
from src.evaluate import load_ground_truth
from src.blocking import run_all_blocking
from src.train import (
    create_train_val_split,
    build_training_pairs,
    train_model,
    find_optimal_threshold,
    save_model,
)
from src.features import compute_pair_features
from src.predict import normalize_dataframe, build_record_dict

# Configuration
SAMPLE_FRAC = 0.05  # Use a small sample for fast interactive running
MAX_NEG_PER_POS = 5

## 2. Load Ground Truth and S1 Data
We load the ground truth matches and the S1 target companies.

In [ ]:
# Load ground truth
gt = load_ground_truth(config.TRAIN_GT)
print(f"{len(gt)} S1 entities in ground truth")

# Create train/val split
train_ids, val_ids = create_train_val_split(gt)

# Load S1
s1_all = pd.read_csv(config.TRAIN_S1, sep="\t", dtype=str, keep_default_na=False)
countries = s1_all["country"].unique()

if SAMPLE_FRAC:
    s1_all = s1_all.sample(frac=SAMPLE_FRAC, random_state=42)
    print(f"Sampled to {len(s1_all)} records")
    
s1_all.head()

## 3. Data Processing Loop (Blocking & Feature Generation)
This loops over countries, loads the databases (S2 & S3), runs the string matching (blocking), and computes the 27 ML features.

In [ ]:
all_X_train = []
all_y_train = []
all_X_val = []
all_y_val = []
all_val_pairs = []

for country in countries:
    print(f"\nProcessing Country: {country}")
    s1_df = s1_all[s1_all["country"] == country].copy()
    s1_entity_ids = set(s1_df["entity_id"])
    
    # Load Pool (S2 + S3)
    s2_df = pd.read_csv(config.TRAIN_S2, sep="\t", dtype=str, keep_default_na=False)
    s2_df = s2_df[s2_df["country"] == country]
    
    s3_df = pd.read_csv(config.TRAIN_S3, sep="\t", dtype=str, keep_default_na=False)
    s3_df = s3_df[s3_df["country"] == country]
    
    pool_df = pd.concat([s2_df, s3_df], ignore_index=True)
    del s2_df, s3_df
    
    # Normalize Names and Addresses
    s1_df = normalize_dataframe(s1_df)
    pool_df = normalize_dataframe(pool_df)
    
    # Run Blocking (Find candidates)
    gt_country = {k: v for k, v in gt.items() if k in s1_entity_ids}
    candidates = run_all_blocking(s1_df, pool_df, country=country, ground_truth=gt_country)
    
    for eid in s1_df["entity_id"]:
        if eid not in candidates:
            candidates[eid] = set()
            
    # Build data dicts
    s1_data = build_record_dict(s1_df)
    pool_data = build_record_dict(pool_df)
    
    train_s1_country = train_ids & s1_entity_ids
    val_s1_country = val_ids & s1_entity_ids
    
    train_pairs = build_training_pairs(train_s1_country, candidates, gt, neg_pos_ratio=MAX_NEG_PER_POS)
    val_pairs = build_training_pairs(val_s1_country, candidates, gt, neg_pos_ratio=MAX_NEG_PER_POS)
    
    # (In a full script, feature computation goes here. Simplified for notebook display)
    print(f"Found {len(train_pairs)} training pairs and {len(val_pairs)} validation pairs for {country}.")